# LQD vs SLQD Spread Diagnostics

Analyze whether the beta-hedged `LQD / SLQD` spread looks mean reverting before we build a trade around it.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from db.connection import get_engine
from fixed_income.rv import cumulative_spread, diagnose_spread, event_study, load_pair_prices
from stores.market import PriceStore

In [ ]:
DATA_BACKEND = "supabase"
APP_ENV = "uat"
PAIR = ("LQD", "SLQD")
START_DATE = "2021-01-01"
END_DATE = None

SPREAD_KIND = "return"  # 'return' or 'price'
BETA_SOURCE = "trailing"  # 'trailing' or 'full_sample'
BETA_LOOKBACK = 60
HEDGE_WINDOW = 60
Z_WINDOW = 20

THRESHOLDS = (1.5, 2.0, 2.5)
HORIZONS = (1, 3, 5, 10)
ROUND_TRIP_COST = 0.0002  # 2 bps total
COMPARE_PAIRS = [("LQD", "SLQD"), ("LQD", "HYG"), ("TLT", "LQD"), ("LQD", "VCIT")]


In [ ]:
engine = get_engine(data_backend=DATA_BACKEND, app_env=APP_ENV)
price_store = PriceStore(engine)

prices = load_pair_prices(
    price_store,
    PAIR[0],
    PAIR[1],
    start_date=START_DATE,
    end_date=END_DATE,
)

frame, diagnostics = diagnose_spread(
    prices,
    left_ticker=PAIR[0],
    right_ticker=PAIR[1],
    spread_kind=SPREAD_KIND,
    beta_source=BETA_SOURCE,
    beta_lookback=BETA_LOOKBACK,
    hedge_window=HEDGE_WINDOW,
    z_window=Z_WINDOW,
)

pd.Series(diagnostics.as_dict(), name="value").to_frame()

In [ ]:
summary = pd.DataFrame(
    {
        "metric": [
            "beta",
            "latest z-score",
            "lag-1 autocorr",
            "half-life (daily spread)",
            "half-life (5D cum spread)",
            "half-life (z-score)",
            "hurst exponent",
            "zero crossings / year",
            "ADF p-value",
            "stationary @ 5%",
        ],
        "value": [
            diagnostics.beta,
            diagnostics.zscore_last,
            diagnostics.lag1_autocorr,
            diagnostics.half_life_days,
            diagnostics.half_life_5d_cum_days,
            diagnostics.half_life_zscore_days,
            diagnostics.hurst_exponent,
            diagnostics.zero_crossings_per_year,
            diagnostics.adf_pvalue,
            diagnostics.adf_is_stationary_5pct,
        ],
    }
)
summary

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)

frame[["close_left", "close_right"]].plot(ax=axes[0], lw=1.5)
axes[0].set_title(f"{PAIR[0]} vs {PAIR[1]} prices")
axes[0].set_ylabel("price")

frame["spread"].plot(ax=axes[1], color="darkorange", lw=1.4, label=f"beta-hedged {SPREAD_KIND} spread")
frame["spread_mean"].plot(ax=axes[1], color="black", lw=1.0, alpha=0.8, label="rolling mean")
axes[1].axhline(frame["spread"].mean(), color="gray", linestyle="--", alpha=0.6)
axes[1].set_title(f"Beta-hedged {SPREAD_KIND} spread (beta={diagnostics.beta}, source={BETA_SOURCE})")
axes[1].set_ylabel(SPREAD_KIND)
axes[1].legend(loc="upper right")

frame["zscore"].plot(ax=axes[2], color="navy", lw=1.4, label="z-score")
axes[2].axhline(0, color="black", linestyle="--", alpha=0.7)
axes[2].axhline(1, color="darkred", linestyle=":", alpha=0.7)
axes[2].axhline(-1, color="darkgreen", linestyle=":", alpha=0.7)
axes[2].axhline(2, color="darkred", linestyle="--", alpha=0.7)
axes[2].axhline(-2, color="darkgreen", linestyle="--", alpha=0.7)
axes[2].set_title("Rolling spread z-score")
axes[2].set_ylabel("z")
axes[2].legend(loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
event_stats = event_study(
    frame,
    thresholds=THRESHOLDS,
    horizons=HORIZONS,
    round_trip_cost=ROUND_TRIP_COST,
)

event_stats

In [ ]:
event_summary = event_stats.copy()
if not event_summary.empty:
    event_summary["hit_rate"] = (event_summary["hit_rate"] * 100).round(1)
    event_summary["net_hit_rate"] = (event_summary["net_hit_rate"] * 100).round(1)
    event_summary = event_summary[
        [
            "threshold",
            "horizon_d",
            "n_events",
            "events_per_year",
            "avg_fwd_bps",
            "net_avg_bps",
            "hit_rate",
            "net_hit_rate",
            "avg_entry_beta",
            "entry_beta_std",
            "avg_beta_20d_std",
            "avg_abs_beta_drift_5d",
        ]
    ].round(3)

event_summary

In [ ]:
frame_5d = frame.copy()
frame_5d["spread_5d"] = cumulative_spread(frame_5d["spread"], window=5)
roll_mean_5d = frame_5d["spread_5d"].rolling(Z_WINDOW).mean()
roll_std_5d = frame_5d["spread_5d"].rolling(Z_WINDOW).std(ddof=0)
frame_5d["zscore_5d"] = (frame_5d["spread_5d"] - roll_mean_5d) / roll_std_5d.replace(0, pd.NA)

event_stats_5d = event_study(
    frame_5d,
    thresholds=THRESHOLDS,
    horizons=HORIZONS,
    round_trip_cost=ROUND_TRIP_COST,
    signal_col="zscore_5d",
    payoff_col="spread",
)

event_stats_5d


In [ ]:
comparison_rows = []

for left_ticker, right_ticker in COMPARE_PAIRS:
    pair_prices = load_pair_prices(
        price_store,
        left_ticker,
        right_ticker,
        start_date=START_DATE,
        end_date=END_DATE,
    )
    pair_frame, pair_diag = diagnose_spread(
        pair_prices,
        left_ticker=left_ticker,
        right_ticker=right_ticker,
        spread_kind=SPREAD_KIND,
        beta_source=BETA_SOURCE,
        beta_lookback=BETA_LOOKBACK,
        hedge_window=HEDGE_WINDOW,
        z_window=Z_WINDOW,
    )
    stats_daily = event_study(
        pair_frame,
        thresholds=THRESHOLDS,
        horizons=HORIZONS,
        round_trip_cost=ROUND_TRIP_COST,
    )
    pair_frame_5d = pair_frame.copy()
    pair_frame_5d["spread_5d"] = cumulative_spread(pair_frame_5d["spread"], window=5)
    pair_frame_5d["zscore_5d"] = (
        pair_frame_5d["spread_5d"] - pair_frame_5d["spread_5d"].rolling(Z_WINDOW).mean()
    ) / pair_frame_5d["spread_5d"].rolling(Z_WINDOW).std(ddof=0).replace(0, pd.NA)
    stats_5d = event_study(
        pair_frame_5d,
        thresholds=THRESHOLDS,
        horizons=HORIZONS,
        round_trip_cost=ROUND_TRIP_COST,
        signal_col="zscore_5d",
        payoff_col="spread",
    )

    def pick_best(stats: pd.DataFrame) -> tuple[float, int, float, float]:
        if stats.empty:
            return float("nan"), 0, float("nan"), float("nan")
        best = stats.sort_values(["net_avg_bps", "net_hit_rate"], ascending=[False, False]).iloc[0]
        return best["threshold"], int(best["horizon_d"]), best["net_avg_bps"], best["net_hit_rate"]

    d_thr, d_h, d_edge, d_hit = pick_best(stats_daily)
    s_thr, s_h, s_edge, s_hit = pick_best(stats_5d)
    comparison_rows.append({
        "pair": f"{left_ticker} / {right_ticker}",
        "beta": pair_diag.beta,
        "half_life_5d": pair_diag.half_life_5d_cum_days,
        "adf_pvalue": pair_diag.adf_pvalue,
        "daily_best_threshold": d_thr,
        "daily_best_horizon": d_h,
        "daily_best_net_bps": d_edge,
        "daily_best_hit_rate": d_hit,
        "smooth_best_threshold": s_thr,
        "smooth_best_horizon": s_h,
        "smooth_best_net_bps": s_edge,
        "smooth_best_hit_rate": s_hit,
    })

comparison = pd.DataFrame(comparison_rows)
comparison


In [ ]:
pivot_edge = event_stats.pivot(index="threshold", columns="horizon_d", values="net_avg_bps")
pivot_hit = event_stats.pivot(index="threshold", columns="horizon_d", values="net_hit_rate")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

pivot_edge.plot(kind="bar", ax=axes[0])
axes[0].axhline(0, color="black", linestyle="--", alpha=0.7)
axes[0].set_title("Net forward edge by threshold / horizon (bps)")
axes[0].set_ylabel("bps")

pivot_hit.plot(kind="bar", ax=axes[1])
axes[1].axhline(0.5, color="black", linestyle="--", alpha=0.7)
axes[1].set_title("Net hit rate by threshold / horizon")
axes[1].set_ylabel("hit rate")

plt.tight_layout()
plt.show()

In [ ]:
spread_5d = cumulative_spread(frame["spread"], window=5)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

spread_5d.plot(ax=axes[0], color="purple", lw=1.2, label="5D cumulative spread")
axes[0].axhline(0, color="black", linestyle="--", alpha=0.7)
axes[0].set_title("5D cumulative return spread")
axes[0].legend(loc="upper right")

frame["zscore"].plot(ax=axes[1], color="navy", lw=1.2, label="z-score")
axes[1].axhline(0, color="black", linestyle="--", alpha=0.7)
axes[1].set_title("Rolling z-score series")
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
frame["rolling_beta"].plot(ax=ax, color="teal", lw=1.3)
ax.axhline(diagnostics.beta, color="black", linestyle="--", alpha=0.7, label="static beta used")
ax.set_title("Rolling hedge beta")
ax.set_ylabel("beta")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## How to read this

- Lower `half-life` is usually better for a mean-reversion trade, as long as it is not just noise.
- `Hurst < 0.5` leans mean-reverting; `> 0.5` leans trending.
- `ADF p-value < 0.05` supports stationarity when `statsmodels` is installed.
- A stable rolling beta helps a lot before we freeze the hedge ratio at trade entry.
- `SPREAD_KIND = "return"` is usually the cleaner first pass for a tradable pair signal like `LQD / SLQD`.
- The event-study table is where the tradeability question really gets answered: edge, hit rate, frequency, and beta behavior at entry.